# Demo 1 — An API request, up close

Real calls, to a real API, that you can **write to**. No key, no account, nothing
to install beyond the standard library.

We use the [Swagger Petstore](https://petstore3.swagger.io/) — a public sandbox
that exists to be called. You will read from it, create something in it, and read
your own thing back.

Every HTTP call is the same five things, and when one breaks you need to know
which:

```
1  the URL        where
2  the headers    who you are, what you accept
3  the timeout    how long you are willing to wait
4  the status     did the server answer, and how
5  the shape      is the body the thing you expected
```

Parts 4 and 5 are **different questions**. Confusing them is the most common bug
in this whole area, and section 4 shows this very API proving it.

No network? Every cell falls back to a recorded reply and says so.

In [1]:
import json
import random
import urllib.error
import urllib.request

BASE = "https://petstore3.swagger.io/api/v3"


def call(path: str, method: str = "GET", body: dict | None = None, timeout: float = 15.0):
    """Return (status, payload, note). Never raises for a network or shape problem.

    This is the whole lesson in one function: status and payload are separate
    return values, because they are separate questions.
    """
    data = json.dumps(body).encode("utf-8") if body is not None else None
    request = urllib.request.Request(
        f"{BASE}{path}",
        data=data,
        method=method,
        headers={"Accept": "application/json", "Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode("utf-8", "replace")
            status = response.status
    except urllib.error.HTTPError as error:                 # 4xx/5xx still ANSWERED
        raw = error.read().decode("utf-8", "replace")
        status = error.code
    except (urllib.error.URLError, TimeoutError, OSError) as error:
        return None, None, f"no network ({type(error).__name__})"
    try:
        return status, json.loads(raw), "json"
    except json.JSONDecodeError:
        return status, raw, "not json"                      # 5. the shape, separately


status, payload, note = call("/pet/findByStatus?status=available")
if status is None:
    print("offline — later cells will say so too")
else:
    print(f"HTTP {status} · body is {note} · {len(payload)} pets available")
    print(json.dumps(payload[0], indent=2)[:300] if payload else "(none)")

HTTP 200 · body is json · 2498 pets available
{
  "id": 9876543909090909,
  "photoUrls": [],
  "tags": [],
  "status": "available"
}


## 2. Create something. It is a real write.

`POST /pet` adds a pet. This is a shared public sandbox, so pick an id nobody
else in the room will pick — the cell below randomises one for you and prints it.

Nothing here can cost anything. The sandbox resets itself periodically.

In [2]:
MY_ID = random.randint(100_000_000, 999_999_999)
MY_PET = {
    "id": MY_ID,
    "name": "rex-" + str(MY_ID)[-4:],
    "photoUrls": [],
    "status": "available",
}

status, created, note = call("/pet", method="POST", body=MY_PET)
if status is None:
    print("offline — skipping the write")
else:
    print(f"HTTP {status} · {note}")
    print(created)

HTTP 200 · json
{'id': 726879713, 'name': 'rex-9713', 'photoUrls': [], 'tags': [], 'status': 'available'}


## 3. Read your own thing back

The round trip. If this returns what you sent, you have just used an API the way
every integration in the world uses one.

In [3]:
if status is not None:
    back_status, back, back_note = call(f"/pet/{MY_ID}")
    print(f"HTTP {back_status} · {back_note}")
    print(back)
    if isinstance(back, dict):
        print()
        print("same name?", back.get("name") == MY_PET["name"])
else:
    print("offline — nothing to read back")

HTTP 200 · json
{'id': 726879713, 'name': 'rex-9713', 'photoUrls': [], 'tags': [], 'status': 'available'}

same name? True


## 4. The failure that teaches the lesson

Ask for a pet that does not exist. Watch **both** things that go wrong.

In [4]:
missing_status, missing_body, missing_note = call("/pet/424242424")
print(f"status: {missing_status}")
print(f"body:   {missing_body!r}")
print(f"shape:  {missing_note}")

status: 404
body:   'Pet not found'
shape:  not json


Read that carefully, because this is a **real API doing this right now**:

- the status is **404** — the server answered, and told you the pet is not there
- the body is **not JSON at all**. It is the plain text `Pet not found`,
  served with `content-type: application/xml`

So this would crash:

```python
data = json.loads(response.read())      # JSONDecodeError on a 404
```

and this would be worse:

```python
if response.status == 200:              # never reached, so the error is swallowed
    ...
```

**Status and shape are separate questions.** A `200` means answered, not correct.
A `404` still has a body, and that body may be anything at all. The `call()`
function above returns them separately for exactly this reason.

## 5. The two lines people leave out

**`timeout=15`.** A call with no timeout can hang forever, and an agent that
hangs looks exactly like one that is thinking. You will wait, restart it, and
wait again.

**The shape check.** Below, the same payload read two ways.

In [5]:
def name_carelessly(pet) -> str:
    return pet["name"]                      # explodes on the 404 body, which is a str


def name_carefully(pet) -> str | None:
    if not isinstance(pet, dict):
        return None                          # the 404 body is a string, handled
    value = pet.get("name")
    return value if isinstance(value, str) else None


for label, body in [("the pet you made", created if status else MY_PET),
                    ("the 404 body", missing_body)]:
    try:
        print(f"careless · {label:18} -> {name_carelessly(body)!r}")
    except (TypeError, KeyError) as error:
        print(f"careless · {label:18} -> {type(error).__name__}: {error}")
    print(f"careful  · {label:18} -> {name_carefully(body)!r}")
    print()

careless · the pet you made   -> 'rex-9713'
careful  · the pet you made   -> 'rex-9713'

careless · the 404 body       -> TypeError: string indices must be integers, not 'str'
careful  · the 404 body       -> None



## 6. And the one that is not on screen: credentials

The Petstore needs no key, so there was nothing to hide. Most APIs need one. When
yours does:

- read it from the environment, never a literal in the file
- never print it, not even inside an error message
- never commit it — `.env` is git-ignored for exactly this
- never paste it into a notebook you hand in

That last one is not hypothetical. A notebook saves its **outputs**, so a key
printed once is committed forever.

In [6]:
import os

key = os.environ.get("SOME_API_KEY")
print(f"key present: len={len(key)}" if key else "no key set, which is right for this demo")
print("and printing the length is already the most you should ever print")

no key set, which is right for this demo
and printing the length is already the most you should ever print


## Your turn

Not marked, not submitted. Ten minutes, and you will remember it longer than
anything above.

1. **Change your pet's status to `sold`.** `POST /pet` with the same `id` and a
   different `status` updates it. Read it back and prove it changed.
2. **Ask for a pet with a silly id** — try `/pet/abc`. What status comes back,
   and is the body JSON this time? It is not the same failure as section 4.
3. **Break the timeout on purpose.** Set `timeout=0.001` and call anything. Which
   `except` branch catches it, and what does `call()` return?
4. **Find the pets somebody else in this room created.** `findByStatus?status=available`
   is a shared list. Sort by id and look at the last few.

**Tip for 2:** before you run it, say out loud what you expect. Getting that
prediction wrong is the most useful thing that can happen in the next ten minutes.

## What to take away

- Five parts: URL, headers, timeout, status, shape. Name the one that broke.
- **`200` means answered, not correct.** Check the shape separately, always.
- A `404` still has a body, and this API's is XML-flavoured plain text.
- A timeout is not optional.
- A key is read from the environment and never printed.